In [ ]:
# Import the libraries

import random
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Use Apple's GPU if available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.14.1
Device: mps


In [19]:
# Step 3: Load and prepare CIFAR-10

# Image normalization values
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

# Transformations applied to training images
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Validation and test images are not randomly changed
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Download the dataset
train_dataset_aug = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

train_dataset_eval = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=eval_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=eval_transform
)

# Create a reproducible 90% training / 10% validation split
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

indices = torch.randperm(len(train_dataset_aug)).tolist()
val_size = 5000

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_subset = Subset(train_dataset_aug, train_indices)
val_subset = Subset(train_dataset_eval, val_indices)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_dataset))

Training images: 45000
Validation images: 5000
Test images: 10000


In [23]:
# Step 4: Prepare batches of images

BATCH_SIZE = 64

def make_loaders(batch_size=64):
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(BATCH_SIZE)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 704
Validation batches: 79
Test batches: 157


In [21]:
class VGG(nn.Module):
    def __init__(self, num_classes=10, activation_layer=nn.ReLU):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            activation_layer(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            activation_layer(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            activation_layer(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            activation_layer(),

            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [26]:
# Step 7: Training and evaluation functions

criterion = nn.CrossEntropyLoss()


def evaluate(model, data_loader):
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = 100.0 * correct / total
    average_loss = total_loss / total

    return average_loss, accuracy


def train_model(model, train_loader, val_loader,
                optimizer, epochs=10):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_accuracy = 0.0
    best_epoch = 0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()

        correct = 0
        total = 0
        total_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Clear previous gradients
            optimizer.zero_grad()

            # Predict the image classes
            outputs = model(images)

            # Calculate how wrong the predictions are
            loss = criterion(outputs, labels)

            # Calculate gradients and update the model
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_accuracy = 100.0 * correct / total

        val_loss, val_accuracy = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_accuracy)

        # Save the model whenever validation accuracy improves
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch
            best_weights = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}% | "
            f"Validation Loss: {val_loss:.4f} | "
            f"Validation Accuracy: {val_accuracy:.2f}%"
        )

    if best_weights is not None:
        model.load_state_dict(best_weights)

    print(f"\nBest validation accuracy: {best_val_accuracy:.2f}%")
    print(f"Best epoch: {best_epoch}")

    return history, best_val_accuracy, best_epoch

In [ ]:
# Step 8: First experiment — baseline configuration

learning_rate = 0.001
epochs = 10

model = VGG(activation_layer=nn.Sigmoid).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

history, best_accuracy, best_epoch = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=epochs
)

# Save the best model
torch.save(
    model.state_dict(),
    "vgg6_baseline_best.pth"
)

print("Baseline experiment completed.")

Epoch 01/10 | Train Loss: 1.8046 | Train Accuracy: 32.78% | Validation Loss: 2.4062 | Validation Accuracy: 18.06%
Epoch 02/10 | Train Loss: 1.5233 | Train Accuracy: 43.92% | Validation Loss: 1.9182 | Validation Accuracy: 34.62%
Epoch 03/10 | Train Loss: 1.4113 | Train Accuracy: 48.48% | Validation Loss: 1.6421 | Validation Accuracy: 39.08%
Epoch 04/10 | Train Loss: 1.3238 | Train Accuracy: 52.45% | Validation Loss: 2.7726 | Validation Accuracy: 31.96%
Epoch 05/10 | Train Loss: 1.2457 | Train Accuracy: 55.24% | Validation Loss: 3.2764 | Validation Accuracy: 23.34%
Epoch 06/10 | Train Loss: 1.1799 | Train Accuracy: 58.12% | Validation Loss: 2.1041 | Validation Accuracy: 35.44%
Epoch 07/10 | Train Loss: 1.1253 | Train Accuracy: 60.00% | Validation Loss: 4.2747 | Validation Accuracy: 19.92%
Epoch 08/10 | Train Loss: 1.0748 | Train Accuracy: 61.87% | Validation Loss: 3.1159 | Validation Accuracy: 28.32%
Epoch 09/10 | Train Loss: 1.0373 | Train Accuracy: 63.26% | Validation Loss: 4.8671 | Va

In [ ]:
import wandb
print("Logged in:", wandb.login())

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.
wandb: Currently logged in as: cs25m522 (cs25m522-iit-madras) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Logged in: True


In [ ]:
sweep_config = {
    "method": "random",

    "metric": {
        "name": "best_val_acc",
        "goal": "maximize"
    },

    "parameters": {
        "activation": {
            "values": ["ReLU", "Tanh", "SiLU", "GELU"]
        },

        "optimizer": {
            "values": ["SGD", "Nesterov-SGD", "Adam",
                       "Adagrad", "RMSprop", "Nadam"]
        },

        "batch_size": {
            "values": [32, 64, 128]
        },

        "epochs": {
            "values": [5, 10, 20]
        },

        "learning_rate": {
            "values": [0.0001, 0.001, 0.01]
        },

        "weight_decay": {
            "values": [0.0, 0.0001]
        }
    }
}

sweep_id = wandb.sweep(
    sweep_config,
    project="VGG6-CIFAR10"
)

print("Sweep ID:", sweep_id)

Create sweep with ID: 3zdb0uey
Sweep URL: https://forge.coreweave.com/wandb/cs25m522-iit-madras/VGG6-CIFAR10/sweeps/3zdb0uey
Sweep ID: 3zdb0uey


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import random
import pandas as pd

from torch.utils.data import DataLoader

print("Imports loaded successfully!")

Imports loaded successfully!


In [ ]:
def train_sweep():
    
    with wandb.init() as run:

        config = wandb.config

        # Activation function
        activation_functions = {
            "ReLU": nn.ReLU,
            "Tanh": nn.Tanh,
            "SiLU": nn.SiLU,
            "GELU": nn.GELU
        }

        model = VGG(
            activation_layer=activation_functions[config.activation]
        ).to(device)

        # DataLoaders
        train_loader_exp = DataLoader(
            train_loader.dataset,
            batch_size=config.batch_size,
            shuffle=True
        )

        val_loader_exp = DataLoader(
            val_loader.dataset,
            batch_size=config.batch_size,
            shuffle=False
        )

        test_loader_exp = DataLoader(
            test_loader.dataset,
            batch_size=config.batch_size,
            shuffle=False
        )

        # Optimizer
        if config.optimizer == "SGD":
            optimizer = optim.SGD(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Nesterov-SGD":
            optimizer = optim.SGD(
                model.parameters(),
                lr=config.learning_rate,
                momentum=0.9,
                nesterov=True,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Adam":
            optimizer = optim.Adam(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Adagrad":
            optimizer = optim.Adagrad(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "RMSprop":
            optimizer = optim.RMSprop(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Nadam":
            optimizer = optim.NAdam(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0
        best_weights = None

        # Training
        for epoch in range(config.epochs):

            model.train()

            running_loss = 0
            correct = 0
            total = 0

            for images, labels in train_loader_exp:

                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                outputs = model(images)
                loss = criterion(outputs, labels)

                loss.backward()
                optimizer.step()

                running_loss += loss.item() * labels.size(0)

                predictions = outputs.argmax(dim=1)

                correct += (predictions == labels).sum().item()
                total += labels.size(0)

            train_loss = running_loss / total
            train_acc = 100 * correct / total

            # Validation
            model.eval()

            val_loss_total = 0
            val_correct = 0
            val_total = 0

            with torch.no_grad():

                for images, labels in val_loader_exp:

                    images = images.to(device)
                    labels = labels.to(device)

                    outputs = model(images)
                    loss = criterion(outputs, labels)

                    val_loss_total += loss.item() * labels.size(0)

                    predictions = outputs.argmax(dim=1)

                    val_correct += (predictions == labels).sum().item()
                    val_total += labels.size(0)

            val_loss = val_loss_total / val_total
            val_acc = 100 * val_correct / val_total

            if val_acc > best_val_acc:

                best_val_acc = val_acc

                best_weights = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }

            # Log metrics after every epoch
            wandb.log({
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "epoch": epoch + 1
            }, step=epoch + 1)

            print(
                f"Epoch {epoch+1}/{config.epochs} | "
                f"Train Accuracy: {train_acc:.2f}% | "
                f"Validation Accuracy: {val_acc:.2f}%"
            )

        # Restore best model
        if best_weights is not None:
            model.load_state_dict(best_weights)

        test_loss, test_accuracy = evaluate(
            model, test_loader_exp
        )

        wandb.log({
            "best_val_acc": best_val_acc,
            "test_acc": test_accuracy,
            "test_loss": test_loss
        })

        print("Best Validation Accuracy:", best_val_acc)
        print("Test Accuracy:", test_accuracy)

install %pip install nbformat

In [27]:
wandb.agent(
    sweep_id,
    function=train_sweep,
    count=5
)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.
wandb: Agent Starting Run: nesi3m56 with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 64
wandb: 	epochs: 5
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Adam
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


Epoch 1/5 | Train Accuracy: 38.22% | Validation Accuracy: 49.04%
Epoch 2/5 | Train Accuracy: 50.87% | Validation Accuracy: 49.52%
Epoch 3/5 | Train Accuracy: 56.96% | Validation Accuracy: 53.72%


Failed to export metrics batch due to timeout, max retries or shutdown.


Epoch 4/5 | Train Accuracy: 61.17% | Validation Accuracy: 56.42%
Epoch 5/5 | Train Accuracy: 63.82% | Validation Accuracy: 58.04%
Best Validation Accuracy: 58.04
Test Accuracy: 57.03


best_val_acc,▁
epoch,▁▃▅▆█
test_acc,▁
test_loss,▁
train_acc,▁▄▆▇█
train_loss,█▅▃▂▁
val_acc,▁▁▅▇█
val_loss,█▇▅▃▁
best_val_acc,58.04
epoch,5
test_acc,57.03


wandb: Agent Starting Run: em1a9yfg with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 128
wandb: 	epochs: 5
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Nadam
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


... (7 outputs left out)

best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▃▄▅▆▆▆▇▇▇▇▇▇▇██████
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▄▄▅▆▆▆▇▇▇▇█▇▇██▇██
val_loss,█▆▅▅▄▃▃▃▂▂▂▂▁▂▁▁▁▂▁▁
best_val_acc,84.96
epoch,20
test_acc,83.32


wandb: Agent Starting Run: iz7odl0t with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	learning_rate: 0.0001
wandb: 	optimizer: RMSprop
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


Epoch 1/10 | Train Accuracy: 35.69% | Validation Accuracy: 38.36%
Epoch 2/10 | Train Accuracy: 45.14% | Validation Accuracy: 47.84%
Epoch 3/10 | Train Accuracy: 48.76% | Validation Accuracy: 40.16%
Epoch 4/10 | Train Accuracy: 51.68% | Validation Accuracy: 47.94%
Epoch 5/10 | Train Accuracy: 53.61% | Validation Accuracy: 51.42%
Epoch 6/10 | Train Accuracy: 55.12% | Validation Accuracy: 53.18%
Epoch 7/10 | Train Accuracy: 56.54% | Validation Accuracy: 53.40%
Epoch 8/10 | Train Accuracy: 57.58% | Validation Accuracy: 53.66%
Epoch 9/10 | Train Accuracy: 58.66% | Validation Accuracy: 55.42%
Epoch 10/10 | Train Accuracy: 59.77% | Validation Accuracy: 58.08%
Best Validation Accuracy: 58.08
Test Accuracy: 56.8


best_val_acc,▁
epoch,▁▂▃▃▄▅▆▆▇█
test_acc,▁
test_loss,▁
train_acc,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▄▃▂▂▂▁▁
val_acc,▁▄▂▄▆▆▆▆▇█
val_loss,▇▄█▄▃▃▃▃▂▁
best_val_acc,58.08
epoch,10
test_acc,56.8


In [32]:
import wandb
import pandas as pd

api = wandb.Api()

# Fetch the sweep using entity/project/sweep_id
sweep = api.sweep(
    "cs25m522-iit-madras/VGG6-CIFAR10/3zdb0uey"
)

results = []

for run in sweep.runs:
    if run.state == "finished":
        results.append({
            "Run Name": run.name,
            "Run ID": run.id,
            "Validation Accuracy": run.summary.get("best_val_acc"),
            "Test Accuracy": run.summary.get("test_acc"),
            "Activation": run.config.get("activation"),
            "Optimizer": run.config.get("optimizer"),
            "Batch Size": run.config.get("batch_size"),
            "Epochs": run.config.get("epochs"),
            "Learning Rate": run.config.get("learning_rate"),
            "Weight Decay": run.config.get("weight_decay")
        })

df = pd.DataFrame(results)

df = df.sort_values(
    by="Validation Accuracy",
    ascending=False
)

display(df)

,Run Name,Run ID,Validation Accuracy,Test Accuracy,Activation,Optimizer,Batch Size,Epochs,Learning Rate,Weight Decay
3,confused-sweep-27,nrc02sru,84.96,83.32,GELU,Nesterov-SGD,32,20,0.0100,0.0000
1,helpful-sweep-25,em1a9yfg,68.54,67.89,ReLU,Nadam,128,5,0.0010,0.0000
2,eternal-sweep-26,s4nvzzoz,61.56,59.68,Tanh,Adam,128,20,0.0001,0.0001
4,winter-sweep-28,iz7odl0t,58.08,56.80,Tanh,RMSprop,32,10,0.0001,0.0001
0,bright-sweep-24,nesi3m56,58.04,57.03,Tanh,Adam,64,5,0.0010,0.0000


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model = VGG(
    num_classes=10,
    activation_layer=nn.GELU
).to(device)

print("Model created successfully!")

Model created successfully!


In [35]:
batch_size = 32
epochs = 20
learning_rate = 0.01
weight_decay = 0.0

optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,
    nesterov=True,
    weight_decay=weight_decay
)

criterion = nn.CrossEntropyLoss()

print("Optimizer created successfully!")

Optimizer created successfully!


In [36]:
activation_layer = nn.GELU

batch_size = 32
epochs = 20
learning_rate = 0.01
weight_decay = 0.0

optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,
    nesterov=True,
    weight_decay=weight_decay
)

In [37]:
model = VGG(
    num_classes=10,
    activation_layer=nn.GELU
).to(device)

In [38]:
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    nesterov=True,
    weight_decay=0.0
)

criterion = nn.CrossEntropyLoss()

In [40]:
import wandb

api = wandb.Api()

run = api.run(
    "cs25m522-iit-madras/VGG6-CIFAR10/nrc02sru"
)

print(run.config)
print("Best validation accuracy:", run.summary["best_val_acc"])
print("Test accuracy:", run.summary["test_acc"])

{'epochs': 20, 'optimizer': 'Nesterov-SGD', 'activation': 'GELU', 'batch_size': 32, 'weight_decay': 0, 'learning_rate': 0.01}
Best validation accuracy: 84.96
Test accuracy: 83.32


In [41]:
config = run.config

activation_name = config["activation"]
batch_size = config["batch_size"]
epochs = config["epochs"]
learning_rate = config["learning_rate"]
weight_decay = config["weight_decay"]
optimizer_name = config["optimizer"]

In [43]:
import wandb

api = wandb.Api()

run = api.run("cs25m522-iit-madras/VGG6-CIFAR10/nrc02sru")

print("Run name:", run.name)
print("Run ID:", run.id)
print("Configuration:", dict(run.config))
print("Summary:", dict(run.summary))

Run name: confused-sweep-27
Run ID: nrc02sru
Configuration: {'epochs': 20, 'optimizer': 'Nesterov-SGD', 'activation': 'GELU', 'batch_size': 32, 'weight_decay': 0, 'learning_rate': 0.01}
Summary: {'_runtime': 408, '_step': 20, '_timestamp': 1793588174.591608, '_wandb.runtime': 408, 'best_val_acc': 84.96, 'epoch': 20, 'test_acc': 83.32, 'test_loss': 0.5031396927833557, 'train_acc': 86.13333333333334, 'train_loss': 0.4005501673221588, 'val_acc': 84.6, 'val_loss': 0.4534407962799072}
